In [7]:
#  SANITY CHECK v10
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import gc
import os
import math

from unet3d_2 import UNet3D
from luna16_dataset_and_dataloader import create_patch_dataloaders

gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device             = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATCH_DIR          = "/home/jovyan/vol.2/unet_patches_vol.2"   # changed
BATCH_SIZE         = 4
NUM_WORKERS        = 2
POSITIVE_FRACTION  = 0.75
INIT_FEATURES      = 48
DROPOUT            = 0.2
COVARIANCE_DROPOUT = 0.2                                        # new
POS_WEIGHT         = 10.0                                       # changed
USE_AMP            = True


class ChannelAwareFocalLoss(nn.Module):
    def __init__(self, base_pos_weight=10.0, alpha=0.75, gamma=2.0):
        super().__init__()
        self.base_pos_weight = base_pos_weight
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets, channel_importances):
        all_imp  = torch.stack([imp.mean() for imp in channel_importances])
        mean_imp = all_imp.mean()
        dyn_pw   = self.base_pos_weight * (1.0 + mean_imp)
        pw_tensor = torch.tensor([dyn_pw.item()], device=inputs.device)
        bce = F.binary_cross_entropy_with_logits(inputs, targets, pos_weight=pw_tensor, reduction='none')
        pt  = torch.exp(-bce.detach())
        loss = torch.mean(self.alpha * (1 - pt) ** self.gamma * bce)
        return loss, dyn_pw.item()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, predict, target):
        predict = torch.sigmoid(predict).view(-1)
        target  = target.view(-1)
        intersection = (predict * target).sum()
        return 1 - (2. * intersection + self.smooth) / (predict.sum() + target.sum() + self.smooth)

_ca_focal = ChannelAwareFocalLoss(base_pos_weight=POS_WEIGHT)
_dice     = DiceLoss()

def sanity_combined_loss(pred, target, importances):
    pw_static = torch.tensor([POS_WEIGHT], device=pred.device)
    bce       = F.binary_cross_entropy_with_logits(pred, target, pos_weight=pw_static)
    focal, dpw = _ca_focal(pred, target, importances)
    dice       = _dice(pred, target)
    return 0.4 * bce + 0.35 * focal + 0.25 * dice, dpw


def calculate_batch_metrics(predictions, targets, threshold=0.5):
    with torch.no_grad():
        binary = (torch.sigmoid(predictions) > threshold).float().view(-1)
        tgt    = targets.view(-1)
        TP = ((binary == 1) & (tgt == 1)).sum().float()
        FP = ((binary == 1) & (tgt == 0)).sum().float()
        FN = ((binary == 0) & (tgt == 1)).sum().float()
        TN = ((binary == 0) & (tgt == 0)).sum().float()

        recall      = TP / (TP + FN + 1e-8)
        precision   = TP / (TP + FP + 1e-8)
        dice        = (2 * TP) / (2 * TP + FP + FN + 1e-8)
        specificity = TN / (TN + FP + 1e-8)
        npv         = TN / (TN + FN + 1e-8)
        mcc_num     = TP * TN - FP * FN
        mcc_den     = ((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)).sqrt()
        mcc         = mcc_num / (mcc_den + 1e-8)

    return {
        'TP': TP.item(), 'FP': FP.item(), 'FN': FN.item(), 'TN': TN.item(),
        'recall':      recall.item(),
        'precision':   precision.item(),
        'dice':        dice.item(),
        'specificity': specificity.item(),
        'npv':         npv.item(),
        'mcc':         mcc.item(),
    }


print("Loading data for sanity check...")
train_loader, _, _ = create_patch_dataloaders(
    patch_dir=PATCH_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    positive_fraction=POSITIVE_FRACTION,
)

sanity_model = UNet3D(
    input_channels=1,
    output_channels=1,
    init_features=INIT_FEATURES,
    dropout=DROPOUT,
    checkpointing=False,
    covariance_dropout=COVARIANCE_DROPOUT,                      # new
).to(device)

n_params = sum(p.numel() for p in sanity_model.parameters() if p.requires_grad)
print(f"Model params: {n_params/1e6:.2f}M  |  device: {device}")
print(f"covariance_dropout: {COVARIANCE_DROPOUT}  |  pos_weight: {POS_WEIGHT}\n")


N_EPOCHS  = 5
N_BATCHES = 100

checks = {
    'forward_pass':      False,
    'loss_decreasing':   False,
    'importance_moving': False,
    'recall_nonzero':    False,
    'no_oom':            False,
}

# two param groups same as full training
importance_params = [
    p for n, p in sanity_model.named_parameters()
    if 'channel_importance_raw' in n
]
base_params = [
    p for n, p in sanity_model.named_parameters()
    if 'channel_importance_raw' not in n
]

opt = optim.AdamW([
    {'params': base_params,       'lr': 5e-5,  'weight_decay': 1e-5},
    {'params': importance_params, 'lr': 5e-3,  'weight_decay': 0.0},
], betas=(0.9, 0.999), eps=1e-8)

sc = GradScaler('cuda', enabled=USE_AMP)

epoch_losses  = []
epoch_recalls = []

imp_before = torch.cat([
    imp.detach().cpu() for imp in sanity_model.get_all_channel_importances()
])

print("=" * 65)
print(f"SANITY CHECK  v10")
print(f"Running {N_EPOCHS} epochs × {N_BATCHES} batches...")
print(f"Metrics: recall, precision, dice, specificity, NPV, MCC")
print("=" * 65)

try:
    for epoch in range(N_EPOCHS):
        sanity_model.train()
        b_losses = []
        b_TP = b_FP = b_FN = b_TN = 0.0

        for step, batch in enumerate(train_loader):
            if step >= N_BATCHES:
                break

            scans = batch['scan'].to(device, non_blocking=True)
            masks = batch['mask'].to(device, non_blocking=True)

            opt.zero_grad()
            with autocast('cuda', enabled=USE_AMP):
                outputs     = sanity_model(scans)
                importances = sanity_model.get_all_channel_importances()
                loss, dpw   = sanity_combined_loss(outputs, masks, importances)

            if torch.isnan(loss):
                raise ValueError("Loss is NaN — check scaling/normalisation")

            if epoch == 0 and step == 0:
                checks['forward_pass'] = True
                print(f"  [OK] Forward pass  — output shape: {outputs.shape}")
                print(f"       Initial dynamic pos_weight: {dpw:.2f}")
                # confirm covariance dropout is present
                gate = sanity_model.decoder4.attention_gate
                print(f"       Covariance dropout p={gate.cov_dropout.p}")

            sc.scale(loss).backward()
            sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(sanity_model.parameters(), 1.0)
            sc.step(opt)
            sc.update()

            b_losses.append(loss.item())
            m = calculate_batch_metrics(outputs, masks)
            b_TP += m['TP']; b_FP += m['FP']
            b_FN += m['FN']; b_TN += m['TN']

        ep_loss        = sum(b_losses) / len(b_losses)
        ep_recall      = b_TP / (b_TP + b_FN + 1e-8)
        ep_precision   = b_TP / (b_TP + b_FP + 1e-8)
        ep_dice        = (2 * b_TP) / (2 * b_TP + b_FP + b_FN + 1e-8)
        ep_specificity = b_TN / (b_TN + b_FP + 1e-8)
        ep_npv         = b_TN / (b_TN + b_FN + 1e-8)
        ep_mcc_num     = b_TP * b_TN - b_FP * b_FN
        ep_mcc_den     = ((b_TP + b_FP) * (b_TP + b_FN) * (b_TN + b_FP) * (b_TN + b_FN)) ** 0.5
        ep_mcc         = ep_mcc_num / (ep_mcc_den + 1e-8)

        epoch_losses.append(ep_loss)
        epoch_recalls.append(ep_recall)

        imp_means = [f"{i.mean().item():.4f}" for i in sanity_model.get_all_channel_importances()]
        imp_stds  = [f"{i.std().item():.4f}"  for i in sanity_model.get_all_channel_importances()]

        print(f"\n  Epoch {epoch+1}/{N_EPOCHS}")
        print(f"    {'Metric':<14} {'Value':>10}")
        print(f"    {'Loss':<14} {ep_loss:>10.4f}")
        print(f"    {'Recall':<14} {ep_recall:>10.4f}")
        print(f"    {'Precision':<14} {ep_precision:>10.4f}")
        print(f"    {'Dice':<14} {ep_dice:>10.4f}")
        print(f"    {'Specificity':<14} {ep_specificity:>10.4f}")
        print(f"    {'NPV':<14} {ep_npv:>10.4f}")
        print(f"    {'MCC':<14} {ep_mcc:>10.4f}")
        print(f"    imp_means: [{' | '.join(imp_means)}]")
        print(f"    imp_stds:  [{' | '.join(imp_stds)}]")

    checks['no_oom'] = True

    if epoch_losses[-1] < epoch_losses[0]:
        checks['loss_decreasing'] = True

    imp_after = torch.cat([
        imp.detach().cpu() for imp in sanity_model.get_all_channel_importances()
    ])
    delta = (imp_after - imp_before).abs().mean().item()
    if delta > 1e-5:
        checks['importance_moving'] = True

    if epoch_recalls[-1] > 0.001:
        checks['recall_nonzero'] = True

except torch.cuda.OutOfMemoryError:
    print("\n  [FAIL] CUDA OOM")
    checks['no_oom'] = False
except Exception as e:
    print(f"\n  [ERROR] {str(e)}")
    all_passed = False


print("\n" + "=" * 65)
print("SANITY CHECK RESULTS  v10")
print("=" * 65)

messages = {
    'forward_pass':      ("Forward pass runs cleanly + covariance dropout confirmed",
                          "Shape error — check INIT_FEATURES, covariance_dropout param, or input dims"),
    'loss_decreasing':   (f"Loss decreasing  {epoch_losses[0]:.4f} → {epoch_losses[-1]:.4f}",
                          f"Loss NOT decreasing  {epoch_losses[0] if epoch_losses else 0:.4f} → {epoch_losses[-1] if epoch_losses else 0:.4f}"),
    'importance_moving': (f"channel_importance learning  Δ={delta:.6f}",
                          f"channel_importance FROZEN  Δ={delta if 'delta' in locals() else 0:.6f}"),
    'recall_nonzero':    (f"Recall non-zero: {epoch_recalls[-1]:.4f}",
                          f"Recall near zero: {epoch_recalls[-1] if epoch_recalls else 0:.4f}"),
    'no_oom':            ("No OOM error",
                          "OOM detected"),
}

all_passed = True
for key, ok in checks.items():
    tag = "PASS" if ok else "FAIL"
    msg = messages[key][0] if ok else messages[key][1]
    print(f"  [{tag}]  {msg}")
    if not ok:
        all_passed = False

print()
if all_passed:
    print("  ✓ ALL CHECKS PASSED")
    print("  → Safe to run full v10 training")
else:
    print("  ✗ FIX FAILURES ABOVE BEFORE TRAINING")

print("=" * 65)

del sanity_model
gc.collect()
torch.cuda.empty_cache()

Loading data for sanity check...
  TRAIN: 49596 patches  | 30240 positive  | 19356 negative
  VAL  : 7280 patches  | 4392 positive  | 2888 negative
  TEST : 14160 patches  | 8640 positive  | 5520 negative

  Loaders | batch=4 | workers=2 | pos_fraction=0.75
  train=12399 batches | val=1820 | test=3540
Model params: 51.78M  |  device: cuda
covariance_dropout: 0.2  |  pos_weight: 10.0

SANITY CHECK  v10
Running 5 epochs × 100 batches...
Metrics: recall, precision, dice, specificity, NPV, MCC
  [OK] Forward pass  — output shape: torch.Size([4, 1, 96, 96, 96])
       Initial dynamic pos_weight: 15.00
       Covariance dropout p=0.2

  Epoch 1/5
    Metric              Value
    Loss               0.3283
    Recall             0.0112
    Precision          0.0011
    Dice               0.0020
    Specificity        0.9859
    NPV                0.9986
    MCC               -0.0009
    imp_means: [0.5241 | 0.5204 | 0.5111 | 0.4961]
    imp_stds:  [0.0337 | 0.0352 | 0.0305 | 0.0268]

  Epoch 

In [ ]:
import os
print(os.getcwd())

In [6]:
import importlib
import unet3d_2
importlib.reload(unet3d_2)
from unet3d_2 import UNet3D
print("UNet3D reloaded")

UNet3D reloaded


In [5]:
import shutil
import os

CHECKPOINT_DIR = './checkpoints'

if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
    print(f"Deleted {CHECKPOINT_DIR}")
else:
    print("No checkpoint directory found, nothing to delete.")

os.makedirs(f'{CHECKPOINT_DIR}/all_epochs', exist_ok=True)
print(f"Created fresh {CHECKPOINT_DIR}/all_epochs/")
print("Ready to run Cell 2.")

Deleted ./checkpoints
Created fresh ./checkpoints/all_epochs/
Ready to run Cell 2.


In [ ]:
import subprocess
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import os
import gc
from unet3d_2 import UNet3D
from luna16_dataset_and_dataloader import create_patch_dataloaders

try:
    __import__("codecarbon")
    print("codecarbon is already installed.")
except ImportError:
    print("codecarbon not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "codecarbon"])

from codecarbon import EmissionsTracker

gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# LOSS FUNCTIONS

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        bce = nn.BCEWithLogitsLoss(reduction='none')(inputs, targets)
        pt  = torch.exp(-bce)
        return torch.mean(self.alpha * (1 - pt) ** self.gamma * bce)

class ChannelAwareFocalLoss(nn.Module):
    def __init__(self, base_pos_weight: float = 10.0, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.base_pos_weight = base_pos_weight
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets, channel_importances):
        all_imp    = torch.stack([imp.mean() for imp in channel_importances])
        mean_imp   = all_imp.mean()
        dyn_pw     = self.base_pos_weight * (1.0 + mean_imp)
        pos_weight = torch.tensor([dyn_pw.item()], device=inputs.device)
        bce        = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')(inputs, targets)
        pt         = torch.exp(-bce)
        return torch.mean(self.alpha * (1 - pt) ** self.gamma * bce), dyn_pw.item()

class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, predict, target):
        predict      = torch.sigmoid(predict).view(-1)
        target       = target.view(-1)
        intersection = (predict * target).sum()
        return 1 - (2. * intersection + self.smooth) / (
            predict.sum() + target.sum() + self.smooth)

_ca_focal = None
_dice     = DiceLoss()

def combined_loss(pred, target, channel_importances, pos_weight_val, label_smoothing=0.0):
    global _ca_focal
    if _ca_focal is None:
        _ca_focal = ChannelAwareFocalLoss(base_pos_weight=pos_weight_val)

    pos_weight_t = torch.tensor([pos_weight_val], device=pred.device)

    if label_smoothing > 0:
        smooth_target = target * (1 - label_smoothing) + 0.5 * label_smoothing
        bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)(pred, smooth_target)
    else:
        bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)(pred, target)

    focal, dyn_pw = _ca_focal(pred, target, channel_importances)
    dice          = _dice(pred, target)

    return 0.4 * bce + 0.35 * focal + 0.25 * dice, dyn_pw


# METRICS

def calculate_batch_metrics(predictions, targets, threshold=0.5):
    binary = (torch.sigmoid(predictions) > threshold).float().view(-1)
    tgt    = targets.view(-1)
    TP = ((binary == 1) & (tgt == 1)).sum().float()
    FP = ((binary == 1) & (tgt == 0)).sum().float()
    FN = ((binary == 0) & (tgt == 1)).sum().float()
    TN = ((binary == 0) & (tgt == 0)).sum().float()

    recall      = TP / (TP + FN + 1e-8)
    precision   = TP / (TP + FP + 1e-8)
    dice        = (2 * TP) / (2 * TP + FP + FN + 1e-8)
    specificity = TN / (TN + FP + 1e-8)
    npv         = TN / (TN + FN + 1e-8)
    mcc_num     = TP * TN - FP * FN
    mcc_den     = ((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)).sqrt()
    mcc         = mcc_num / (mcc_den + 1e-8)

    return {
        'TP': TP.item(), 'FP': FP.item(), 'FN': FN.item(), 'TN': TN.item(),
        'recall':      recall.item(),
        'precision':   precision.item(),
        'dice':        dice.item(),
        'specificity': specificity.item(),
        'npv':         npv.item(),
        'mcc':         mcc.item(),
    }


# CONFIGURATION
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PATCH_DIR                = "/home/jovyan/vol.2/unet_patches_vol.2"
BATCH_SIZE               = 4
POSITIVE_FRACTION        = 0.75
NUM_WORKERS              = 2
INIT_FEATURES            = 48
DROPOUT                  = 0.2
COVARIANCE_DROPOUT       = 0.2
INITIAL_LR               = 1e-4
IMPORTANCE_LR            = 1e-2
WEIGHT_DECAY             = 1e-5
BETAS                    = (0.9, 0.999)
EPS                      = 1e-8
POS_WEIGHT               = 10.0
LABEL_SMOOTHING          = 0.0
ACCUM_STEPS              = 4
USE_AMP                  = True
GRAD_CLIP                = 1.0
NUM_EPOCHS               = 80
EARLY_STOPPING_PATIENCE  = 15
CHECKPOINT_DIR           = './checkpoints'

os.makedirs(f'{CHECKPOINT_DIR}/all_epochs', exist_ok=True)

print("=" * 65)
print("U-NET TRAINING  v10")
print("=" * 65)
print(f"  patch_dir        : {PATCH_DIR}")
print(f"  batch_size       : {BATCH_SIZE}  (effective {BATCH_SIZE*ACCUM_STEPS} with accum)")
print(f"  optimizer        : AdamW  lr={INITIAL_LR}  importance_lr={IMPORTANCE_LR}  wd={WEIGHT_DECAY}")
print(f"  scheduler        : ReduceLROnPlateau  factor=0.5  patience=3")
print(f"  pos_weight       : {POS_WEIGHT}  (+ dynamic ChannelAware scaling)")
print(f"  focal alpha      : 0.75  |  loss: 0.4*BCE + 0.35*Focal + 0.25*Dice")
print(f"  dropout          : {DROPOUT}  |  covariance_dropout: {COVARIANCE_DROPOUT}  |  AMP: {USE_AMP}")
print(f"  epochs           : {NUM_EPOCHS}  |  patience: {EARLY_STOPPING_PATIENCE}")
print(f"  early stopping   : val_recall  |  threshold: 0.5 (sweep on test after training)")
print(f"  metrics          : recall, precision, dice, specificity, NPV, MCC")
print("=" * 65)


# DATA
print("\nLoading datasets...")
train_loader, val_loader, test_loader = create_patch_dataloaders(
    patch_dir=PATCH_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    positive_fraction=POSITIVE_FRACTION,
)


# MODEL
model = UNet3D(
    input_channels=1,
    output_channels=1,
    init_features=INIT_FEATURES,
    dropout=DROPOUT,
    checkpointing=False,
    covariance_dropout=COVARIANCE_DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel params: {n_params/1e6:.2f}M  |  device: {device}")


# OPTIMIZER — two param groups
importance_params = [
    p for n, p in model.named_parameters()
    if 'channel_importance_raw' in n
]
base_params = [
    p for n, p in model.named_parameters()
    if 'channel_importance_raw' not in n
]

print(f"  Base params:       {sum(p.numel() for p in base_params)/1e6:.2f}M  lr={INITIAL_LR}")
print(f"  Importance params: {sum(p.numel() for p in importance_params)}  lr={IMPORTANCE_LR}")

optimizer = optim.AdamW([
    {'params': base_params,       'lr': INITIAL_LR,    'weight_decay': WEIGHT_DECAY},
    {'params': importance_params, 'lr': IMPORTANCE_LR, 'weight_decay': 0.0},
], betas=BETAS, eps=EPS)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)
scaler = GradScaler(enabled=USE_AMP)


# RESUME FROM CHECKPOINT
RESUME_FROM = f'{CHECKPOINT_DIR}/last_checkpoint.pth'
start_epoch = 0

if os.path.exists(RESUME_FROM):
    print(f"\nResuming from checkpoint: {RESUME_FROM}")
    checkpoint = torch.load(RESUME_FROM, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])

    start_epoch        = checkpoint['epoch']
    best_val_loss      = checkpoint.get('best_val_loss', float('inf'))
    best_val_recall    = checkpoint.get('best_val_recall', 0.0)
    best_balanced_dice = checkpoint.get('best_balanced_dice', 0.0)

    train_losses      = checkpoint['train_losses']
    val_losses        = checkpoint['val_losses']
    train_recalls     = checkpoint['train_recalls']
    val_recalls       = checkpoint['val_recalls']
    train_precisions  = checkpoint['train_precisions']
    val_precisions    = checkpoint['val_precisions']
    train_dices       = checkpoint['train_dices']
    val_dices         = checkpoint['val_dices']
    learning_rates    = checkpoint['learning_rates']

    epochs_no_improve = 0
    for r in reversed(val_recalls):
        if r < best_val_recall:
            epochs_no_improve += 1
        else:
            break

    print(f"  Resumed at epoch {start_epoch + 1}")
    print(f"  best_val_loss={best_val_loss:.4f}  best_recall={best_val_recall:.4f}")
    print(f"  No-improve streak: {epochs_no_improve}")
    print(f"  NOTE: optimizer/scheduler starting fresh with new LRs")
else:
    print("\nNo checkpoint found — starting fresh.")
    best_val_loss      = float('inf')
    best_val_recall    = 0.0
    best_balanced_dice = 0.0
    epochs_no_improve  = 0
    train_losses, val_losses         = [], []
    train_recalls, val_recalls       = [], []
    train_precisions, val_precisions = [], []
    train_dices, val_dices           = [], []
    learning_rates                   = []

print("\nSTARTING TRAINING\n")


# CODECARBON
tracker = EmissionsTracker(
    project_name="unet3d_luna16",
    log_level="error",
    save_to_file=False,
)
tracker.start()
training_start_time = time.time()


# TRAINING LOOP
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    train_TP = train_FP = train_FN = train_TN = 0.0
    epoch_dyn_pw = []

    current_lr = optimizer.param_groups[0]['lr']
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train] lr={current_lr:.2e}")
    optimizer.zero_grad()

    for step, batch in enumerate(pbar):
        scans = batch['scan'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)

        with autocast(enabled=USE_AMP):
            outputs      = model(scans)
            importances  = model.get_all_channel_importances()
            loss, dyn_pw = combined_loss(outputs, masks, importances, POS_WEIGHT, LABEL_SMOOTHING)
            loss         = loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        epoch_dyn_pw.append(dyn_pw)

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUM_STEPS

        with torch.no_grad():
            m = calculate_batch_metrics(outputs, masks)
            train_TP += m['TP'];  train_FP += m['FP']
            train_FN += m['FN'];  train_TN += m['TN']

        pbar.set_postfix({'loss': f"{loss.item() * ACCUM_STEPS:.4f}"})

    avg_train_loss    = train_loss / len(train_loader)
    train_recall      = train_TP / (train_TP + train_FN + 1e-8)
    train_precision   = train_TP / (train_TP + train_FP + 1e-8)
    train_dice        = (2 * train_TP) / (2 * train_TP + train_FP + train_FN + 1e-8)
    train_specificity = train_TN / (train_TN + train_FP + 1e-8)
    train_npv         = train_TN / (train_TN + train_FN + 1e-8)
    train_mcc_num     = train_TP * train_TN - train_FP * train_FN
    train_mcc_den     = ((train_TP + train_FP) * (train_TP + train_FN) *
                         (train_TN + train_FP) * (train_TN + train_FN)) ** 0.5
    train_mcc         = train_mcc_num / (train_mcc_den + 1e-8)
    avg_dyn_pw        = sum(epoch_dyn_pw) / len(epoch_dyn_pw)

    model.eval()
    val_loss = 0.0
    val_TP = val_FP = val_FN = val_TN = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]  "):
            scans = batch['scan'].to(device, non_blocking=True)
            masks = batch['mask'].to(device, non_blocking=True)
            with autocast(enabled=USE_AMP):
                outputs     = model(scans)
                importances = model.get_all_channel_importances()
                v_loss, _   = combined_loss(outputs, masks, importances, POS_WEIGHT, LABEL_SMOOTHING)
                val_loss   += v_loss.item()
            m = calculate_batch_metrics(outputs, masks)
            val_TP += m['TP'];  val_FP += m['FP']
            val_FN += m['FN'];  val_TN += m['TN']

    avg_val_loss    = val_loss / len(val_loader)
    val_recall      = val_TP / (val_TP + val_FN + 1e-8)
    val_precision   = val_TP / (val_TP + val_FP + 1e-8)
    val_dice        = (2 * val_TP) / (2 * val_TP + val_FP + val_FN + 1e-8)
    val_specificity = val_TN / (val_TN + val_FP + 1e-8)
    val_npv         = val_TN / (val_TN + val_FN + 1e-8)
    val_mcc_num     = val_TP * val_TN - val_FP * val_FN
    val_mcc_den     = ((val_TP + val_FP) * (val_TP + val_FN) *
                       (val_TN + val_FP) * (val_TN + val_FN)) ** 0.5
    val_mcc         = val_mcc_num / (val_mcc_den + 1e-8)

    scheduler.step(avg_val_loss)
    current_lr_after = optimizer.param_groups[0]['lr']

    train_losses.append(avg_train_loss);       val_losses.append(avg_val_loss)
    train_recalls.append(train_recall);        val_recalls.append(val_recall)
    train_precisions.append(train_precision);  val_precisions.append(val_precision)
    train_dices.append(train_dice);            val_dices.append(val_dice)
    learning_rates.append(current_lr)

    # SUMMARY
    imp_tensors = model.get_all_channel_importances()
    imp_means   = [f"{i.mean().item():.3f}" for i in imp_tensors]
    imp_maxes   = [f"{i.max().item():.3f}"  for i in imp_tensors]
    imp_stds    = [f"{i.std().item():.4f}"  for i in imp_tensors]

    print(f"\nEPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"  {'Metric':<14} {'Train':>10} {'Val':>10} {'Δ':>10}")
    print(f"  {'Loss':<14} {avg_train_loss:>10.4f} {avg_val_loss:>10.4f} {avg_val_loss-avg_train_loss:>+10.4f}")
    print(f"  {'Recall':<14} {train_recall:>10.4f} {val_recall:>10.4f} {val_recall-train_recall:>+10.4f}")
    print(f"  {'Precision':<14} {train_precision:>10.4f} {val_precision:>10.4f} {val_precision-train_precision:>+10.4f}")
    print(f"  {'Dice':<14} {train_dice:>10.4f} {val_dice:>10.4f} {val_dice-train_dice:>+10.4f}")
    print(f"  {'Specificity':<14} {train_specificity:>10.4f} {val_specificity:>10.4f} {val_specificity-train_specificity:>+10.4f}")
    print(f"  {'NPV':<14} {train_npv:>10.4f} {val_npv:>10.4f} {val_npv-train_npv:>+10.4f}")
    print(f"  {'MCC':<14} {train_mcc:>10.4f} {val_mcc:>10.4f} {val_mcc-train_mcc:>+10.4f}")
    print(f"  LR (base): {current_lr:.2e} → {current_lr_after:.2e}  |  LR (importance): {optimizer.param_groups[1]['lr']:.2e}")
    print(f"  Dynamic pos_weight (avg): {avg_dyn_pw:.2f}")
    print(f"  Importance mean  (dec4→dec1): {' | '.join(imp_means)}")
    print(f"  Importance max   (dec4→dec1): {' | '.join(imp_maxes)}")
    print(f"  Importance std   (dec4→dec1): {' | '.join(imp_stds)}")

    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        resv  = torch.cuda.memory_reserved() / 1e9
        print(f"  GPU: {alloc:.1f}GB alloc / {resv:.1f}GB reserved")
        if resv > 20:
            print(f"  WARNING: High GPU memory ({resv:.1f}GB)")
        torch.cuda.reset_peak_memory_stats()

    # CHECKPOINT
    checkpoint = {
        'epoch':                  epoch + 1,
        'model_state_dict':       model.state_dict(),
        'optimizer_state_dict':   optimizer.state_dict(),
        'scheduler_state_dict':   scheduler.state_dict(),
        'scaler_state_dict':      scaler.state_dict(),
        'train_losses':           train_losses,
        'val_losses':             val_losses,
        'train_recalls':          train_recalls,
        'val_recalls':            val_recalls,
        'train_precisions':       train_precisions,
        'val_precisions':         val_precisions,
        'train_dices':            train_dices,
        'val_dices':              val_dices,
        'learning_rates':         learning_rates,
        'best_val_loss':          best_val_loss,
        'best_val_recall':        best_val_recall,
        'best_balanced_dice':     best_balanced_dice,
        'config': {
            'version':              'v10',
            'batch_size':           BATCH_SIZE,
            'accum_steps':          ACCUM_STEPS,
            'optimizer':            'AdamW',
            'initial_lr':           INITIAL_LR,
            'importance_lr':        IMPORTANCE_LR,
            'weight_decay':         WEIGHT_DECAY,
            'betas':                BETAS,
            'scheduler':            'ReduceLROnPlateau',
            'pos_weight':           POS_WEIGHT,
            'focal_alpha':          0.75,
            'loss_weights':         '0.4*BCE + 0.35*ChannelAwareFocal + 0.25*Dice',
            'dropout':              DROPOUT,
            'covariance_dropout':   COVARIANCE_DROPOUT,
            'patch_size':           (96, 96, 96),
            'amp':                  USE_AMP,
            'device':               str(device),
        },
    }

    torch.save(checkpoint, f'{CHECKPOINT_DIR}/all_epochs/epoch_{epoch+1:02d}.pth')
    torch.save(checkpoint, f'{CHECKPOINT_DIR}/last_checkpoint.pth')

    # EARLY STOPPING — tracks val_recall
    if val_recall > best_val_recall:
        best_val_recall = val_recall
        epochs_no_improve = 0
        torch.save(checkpoint, f'{CHECKPOINT_DIR}/best_recall.pth')
        print(f"  ✓ Best val recall: {val_recall:.4f}  (loss={avg_val_loss:.4f}  precision={val_precision:.4f}  dice={val_dice:.4f})")
    else:
        epochs_no_improve += 1

    # still save best loss separately for reference
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(checkpoint, f'{CHECKPOINT_DIR}/best_loss.pth')
        print(f"  ✓ Best val loss:   {avg_val_loss:.4f}  (recall={val_recall:.4f}  precision={val_precision:.4f})")

    if val_dice > best_balanced_dice:
        best_balanced_dice = val_dice

    print(f"  No-improve streak: {epochs_no_improve}/{EARLY_STOPPING_PATIENCE}")

    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f"\nEARLY STOPPING after {epoch+1} epochs.")
        break


# CODECARBON
emissions      = tracker.stop() * 1000
energy_wh      = tracker._total_energy.kWh * 1000
execution_time = time.time() - training_start_time

print(f"\n~ : ~ consumption measured through CodeCarbon ~ : ~")
print(f"Python energy:         {energy_wh:.6f} Wh")
print(f"Python emissions:      {emissions:.6f} CO₂eq grams")
print(f"Python execution time: {execution_time:.2f} seconds  ({execution_time/3600:.2f} hours)")


# FINAL SUMMARY
print("\n" + "=" * 65)
print("TRAINING COMPLETE")
print(f"  Best val loss      : {best_val_loss:.4f}")
print(f"  Best val recall    : {best_val_recall:.4f}")
print(f"  Best balanced dice : {best_balanced_dice:.4f}")
print(f"\nCheckpoints in: {CHECKPOINT_DIR}/")
print(f"  best_recall.pth     – highest val recall (primary)")
print(f"  best_loss.pth       – lowest val loss  (reference)")
print(f"  last_checkpoint.pth – resume from here after kernel dies")
print("=" * 65)

codecarbon not found. Installing...
  Using cached authlib-1.7.2-py2.py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_ml_py-13.595.45-py3-none-any.whl.metadata (9.7 kB)
  Using cached rapidfuzz-3.14.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (12 kB)
  Using cached questionary-2.1.1-py3-none-any.whl.metadata (5.4 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached pycountry-26.2.16-py3-none-any.whl.metadata (12 kB)
  Using cached joserfc-1.6.5-py3-none-any.whl.metadata (3.2 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
Using cached authlib-1.7.2-py2.py3-none-any.whl (259 kB)
Using cached joserfc-1.6.5-py3-none-any.whl (70 kB)
Using cac

/tmp/ipykernel_49/777801182.py:209: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)



Resuming from checkpoint: ./checkpoints/last_checkpoint.pth


[codecarbon WARNING @ 13:24:12] Multiple instances of codecarbon are allowed to run at the same time.


  Resumed at epoch 9
  best_val_loss=0.1445  best_recall=0.8500
  No-improve streak: 2
  NOTE: optimizer/scheduler starting fresh with new LRs

STARTING TRAINING



Epoch 9/80 [Train] lr=1.00e-04:   0%|          | 0/12399 [00:00<?, ?it/s]/tmp/ipykernel_49/777801182.py:288: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
Epoch 9/80 [Val]  :   0%|          | 0/1820 [00:00<?, ?it/s]/tmp/ipykernel_49/777801182.py:333: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
Epoch 9/80 [Val]  : 100%|██████████| 1820/1820 [06:56<00:00,  4.37it/s]



EPOCH 9/80
  Metric              Train        Val          Δ
  Loss               0.0693     0.1446    +0.0753
  Recall             0.8330     0.8124    -0.0206
  Precision          0.7804     0.8317    +0.0513
  Dice               0.8059     0.8219    +0.0161
  Specificity        0.9997     0.9998    +0.0001
  NPV                0.9998     0.9998    +0.0000
  MCC                0.8060     0.8218    +0.0158
  LR (base): 1.00e-04 → 1.00e-04  |  LR (importance): 1.00e-02
  Dynamic pos_weight (avg): 15.25
  Importance mean  (dec4→dec1): 0.251 | 0.414 | 0.618 | 0.813
  Importance max   (dec4→dec1): 0.979 | 0.981 | 0.994 | 1.000
  Importance std   (dec4→dec1): 0.3034 | 0.2926 | 0.2916 | 0.2470
  GPU: 1.3GB alloc / 18.3GB reserved
  No-improve streak: 3/15


Epoch 10/80 [Train] lr=1.00e-04:   2%|▏         | 193/12399 [01:37<1:42:18,  1.99it/s, loss=0.0406]